In [ ]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"clinvar_clean.parquet"
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = os.path.dirname(os.path.abspath(INPUT_PARQUET)) or "."
CACHE_DIR = r"vep_cache" 
RESOURCES_DIR = r"vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{os.path.abspath(WORKDIR)}:{DOCKER_WORKDIR}",
        "-v", f"{os.path.abspath(CACHE_DIR)}:{DOCKER_CACHE}",
        "-v", f"{os.path.abspath(RESOURCES_DIR)}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_WORKDIR}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", f"PhyloP,file={DOCKER_RESOURCES}/hg38.phyloP100way.bw",
        "--plugin", f"phastCons,file={DOCKER_RESOURCES}/hg38.phastCons100way.bw",
        "--plugin", "LoFtool",
        "--plugin", "LOFTEE",
        "--plugin", f"SpliceAI,snv={DOCKER_RESOURCES}/spliceai_scores.raw.snv.hg38.vcf.gz",
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,LoFtool,"
            "phyloP100way_vertebrate,phyloP30way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons30way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,SiPhy_29way_logOdds,"
            "SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")
    return merged_chunk

In [ ]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

In [ ]:
# Chuẩn bị danh sách chứa các df đã xử lý
processed_chunks = []

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...", end="\r")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP
    annotated_chunk = process_chunk(current_chunk, i+1)
    processed_chunks.append(annotated_chunk)

print("\n[*] Hoàn tất chạy VEP. Đang tổng hợp dữ liệu...")

# Gộp toàn bộ chunks
final_df = pd.concat(processed_chunks, ignore_index=True)

# Xóa file tạm để dọn dẹp không gian
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"[SUCCESS] Pipeline hoàn tất! Kích thước dữ liệu đầu ra: {final_df.shape}")

In [ ]:
final_df['Consequence'].value_counts()

Consequence
missense_variant                                                                                            2046055
synonymous_variant                                                                                           633594
intron_variant                                                                                               269068
downstream_gene_variant                                                                                      120652
splice_polypyrimidine_tract_variant,intron_variant                                                            80551
                                                                                                             ...   
splice_donor_variant,coding_sequence_variant,3_prime_UTR_variant                                                  1
splice_acceptor_variant,splice_donor_5th_base_variant,splice_polypyrimidine_tract_variant,intron_variant          1
splice_region_variant,intron_variant,NMD_transcript_variant 

In [ ]:
final_df.isnull().sum()

AlleleID                     0
Type                         0
Name                         0
GeneID                       0
GeneSymbol                 773
HGNC_ID                   5232
ClinicalSignificance         0
ClinSigSimple                0
PhenotypeIDS             56639
PhenotypeList             2973
Origin                       0
OriginSimple                 0
Assembly                     0
ChromosomeAccession          0
CHROM                        0
Start                        0
Stop                         0
ReviewStatus                 0
VariationID                  0
POS                          0
REF                          0
ALT                          0
Uploaded_variation           0
Consequence             258320
HGVSc                   258320
Feature                 258320
Protein_id              258320
Protein_position        258320
Amino_acids             258323
Codons                  258320
dtype: int64

In [ ]:
final_df = final_df.dropna(subset=['Consequence'])
final_df.isnull().sum()

AlleleID                    0
Type                        0
Name                        0
GeneID                      0
GeneSymbol                635
HGNC_ID                  4512
ClinicalSignificance        0
ClinSigSimple               0
PhenotypeIDS            51711
PhenotypeList            2652
Origin                      0
OriginSimple                0
Assembly                    0
ChromosomeAccession         0
CHROM                       0
Start                       0
Stop                        0
ReviewStatus                0
VariationID                 0
POS                         0
REF                         0
ALT                         0
Uploaded_variation          0
Consequence                 0
HGVSc                       0
Feature                     0
Protein_id                  0
Protein_position            0
Amino_acids                 3
Codons                      0
dtype: int64

In [ ]:
final_df = final_df.dropna(subset=['HGVSc'])
final_df.isnull().sum()

AlleleID                    0
Type                        0
Name                        0
GeneID                      0
GeneSymbol                635
HGNC_ID                  4512
ClinicalSignificance        0
ClinSigSimple               0
PhenotypeIDS            51711
PhenotypeList            2652
Origin                      0
OriginSimple                0
Assembly                    0
ChromosomeAccession         0
CHROM                       0
Start                       0
Stop                        0
ReviewStatus                0
VariationID                 0
POS                         0
REF                         0
ALT                         0
Uploaded_variation          0
Consequence                 0
HGVSc                       0
Feature                     0
Protein_id                  0
Protein_position            0
Amino_acids                 3
Codons                      0
dtype: int64

In [ ]:
final_df.to_parquet(r'clinvar_vep_mapping.parquet', index=False)